In [1]:
from gitsource import GithubRepositoryDataReader

# Initialize the reader for the course repository
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

# Fetch files from GitHub
files = reader.read()

# Parse the markdown files into a list of dictionaries
documents = []
for file in files:
    doc = file.parse()
    documents.append(doc)

# Answer Question 1
print(f"Total lesson pages: {len(documents)}")

Total lesson pages: 72


In [2]:
from minsearch import Index

# Initialize the index with the exact fields requested
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

# Fit the index using the documents variable from Step 1
index.fit(documents)

# Define the homework query
query = "How does the agentic loop keep calling the model until it stops?"

# Execute the search
results = index.search(
    query=query,
    num_results=5
)

# Output the filename of the top result
if results:
    print(f"First result filename: {results[0]['filename']}")
else:
    print("No results found. Verify your documents list is populated.")

First result filename: 01-agentic-rag/lessons/14-agentic-loop.md


In [8]:
import httpx
from google import genai
from google.genai import types

# 1. Initialize the client using HttpOptions for custom engine args
client = genai.Client(
    http_options=types.HttpOptions(
        client_args={
            "transport": httpx.HTTPTransport(local_address="0.0.0.0"),
            "timeout": 10.0
        }
    )
)

# 2. Execute the request
query = "How does the agentic loop keep calling the model until it stops?"
results = index.search(query=query, num_results=5)

context_template = "Filename: {filename}\nContent: {content}"
context_pieces = [context_template.format(filename=doc['filename'], content=doc['content']) for doc in results]
context = "\n\n".join(context_pieces)

prompt_template = """
You are a teaching assistant. Answer the QUESTION based on the CONTEXT provided.

CONTEXT:
{context}

QUESTION:
{query}
""".strip()

prompt = prompt_template.format(context=context, query=query)

try:
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    print(f"Gemini Prompt Tokens: {response.usage_metadata.prompt_token_count}")
except Exception as e:
    print(f"Execution failed: {e}")

Gemini Prompt Tokens: 7922


In [9]:
from gitsource import chunk_documents

# Execute chunking with the exact parameters specified
chunks = chunk_documents(documents, size=2000, step=1000)

# Output total count to verify against the options
print(f"Total chunks generated: {len(chunks)}")

Total chunks generated: 295


In [10]:
import httpx
from google import genai
from google.genai import types
from minsearch import Index
from gitsource import chunk_documents

# 1. Re-index using the 295 chunks from Q4
chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
chunk_index.fit(chunks)

# 2. Search using the identical query
query = "How does the agentic loop keep calling the model until it stops?"
chunk_results = chunk_index.search(query=query, num_results=5)

# 3. Rebuild context from the chunk matches
context_template = "Filename: {filename}\nContent: {content}"
context_pieces = [context_template.format(filename=doc['filename'], content=doc['content']) for doc in chunk_results]
chunk_context = "\n\n".join(context_pieces)

prompt_template = """
You are a teaching assistant. Answer the QUESTION based on the CONTEXT provided.

CONTEXT:
{context}

QUESTION:
{query}
""".strip()

chunk_prompt = prompt_template.format(context=chunk_context, query=query)

# 4. Initialize the client forcing IPv4
client = genai.Client(
    http_options=types.HttpOptions(
        client_args={
            "transport": httpx.HTTPTransport(local_address="0.0.0.0"),
            "timeout": 10.0
        }
    )
)

# 5. Execute and print the new token count
try:
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=chunk_prompt
    )
    chunk_tokens = response.usage_metadata.prompt_token_count
    print(f"Chunked Version Prompt Tokens: {chunk_tokens}")
    
    # Calculate ratio against your previous Q3 count (~7922)
    ratio = 7922 / chunk_tokens
    print(f"Reduction factor: {ratio:.1f}x fewer tokens")
except Exception as e:
    print(f"Execution failed: {e}")

Chunked Version Prompt Tokens: 2575
Reduction factor: 3.1x fewer tokens


In [11]:
import httpx
from google import genai
from google.genai import types

# 1. Global counter to track how many times the agent triggers a search
search_counter = 0

def search_chunks(query: str) -> str:
    """
    Searches the lesson document chunks index for relevant context using keywords.
    
    Args:
        query: The keyword search term string.
    """
    global search_counter
    search_counter += 1
    print(f"[Tool Call {search_counter}] Searching index for: '{query}'")
    
    # Use the chunk_index built in Q5
    results = chunk_index.search(query=query, num_results=3)
    
    context_template = "Filename: {filename}\nContent: {content}"
    return "\n\n".join([context_template.format(filename=d['filename'], content=d['content']) for d in results])

# 2. Set up the agent instructions and query
system_instruction = (
    "You're a course teaching assistant. Answer the student's question using the search tool. "
    "Make multiple searches with different keywords before answering."
)
user_query = "How does the agentic loop work, and how is it different from plain RAG?"

# 3. Initialize client with IPv4 config
client = genai.Client(
    http_options=types.HttpOptions(
        client_args={
            "transport": httpx.HTTPTransport(local_address="0.0.0.0"),
            "timeout": 15.0
        }
    )
)

# 4. Run the Agentic Loop Natively
try:
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=user_query,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            tools=[search_chunks], # Register the local python function as a tool
        )
    )
    
    print("\n--- Final Agent Answer ---")
    print(response.text)
    print(f"\nTotal Search Tool Calls: {search_counter}")

except Exception as e:
    print(f"Agent loop failed: {e}")

[Tool Call 1] Searching index for: 'agentic loop'
[Tool Call 2] Searching index for: 'plain RAG'

--- Final Agent Answer ---
The agentic loop is a dynamic and iterative process where an AI agent interacts with an LLM, executes tools, and incorporates the results to reach a final answer. It works by:

1.  **Calling the LLM:** The LLM receives a prompt or query.
2.  **Executing Tool Calls:** Based on the prompt, the LLM may decide to use one or more external tools (e.g., search engines, code interpreters).
3.  **Sending Results Back:** The outputs from these tool calls are then fed back to the LLM.
4.  **Iteration:** This process repeats in a `while` loop. The agent continues to call the LLM and execute tools, adapting its actions based on new information, until it arrives at a final answer without needing further tool calls.
5.  **Conversation History Management:** The agentic loop maintains conversation history, allowing the LLM to understand context from previous turns and make more i

In [1]:
import httpx
from google import genai
from google.genai import types
from toyaikit import Agent

# 1. Reuse the tracking counter and tool from before
search_counter = 0

def search_chunks(query: str) -> str:
    """
    Searches the lesson document chunks index for relevant context using keywords.
    
    Args:
        query: The keyword search term string.
    """
    global search_counter
    search_counter += 1
    print(f"[ToyAIKit Call {search_counter}] Searching for: '{query}'")
    results = chunk_index.search(query=query, num_results=3)
    context_template = "Filename: {filename}\nContent: {content}"
    return "\n\n".join([context_template.format(filename=d['filename'], content=d['content']) for d in results])

# 2. Configure the underlying IPv4 client function for ToyAIKit if it allows customization
# Note: ToyAIKit usually wraps the client or relies on environment variables. 
# We define a custom generation function to pass into the Agent if needed, 
# or we let ToyAIKit call the model.

system_instruction = (
    "You're a course teaching assistant. Answer the student's question using the search tool. "
    "Make multiple searches with different keywords before answering."
)
user_query = "How does the agentic loop work, and how is it different from plain RAG?"

# 3. Initialize ToyAIKit Agent
# ToyAIKit typically expects an OpenAI-like interface or a model name.
# If ToyAIKit lacks direct Gemini integration, it will use the default OpenAI client layout.
try:
    agent = Agent(
        instructions=system_instruction,
        tools=[search_chunks],
        model="gemini/gemini-2.5-flash" # ToyAIKit uses litellm style string formats
    )

    response = agent.run(user_query)
    print("\n--- ToyAIKit Agent Answer ---")
    print(response)
    print(f"\nTotal ToyAIKit Search Tool Calls: {search_counter}")

except Exception as e:
    print(f"ToyAIKit execution failed or requires OpenAI API key layout: {e}")

ImportError: cannot import name 'agent' from 'toyaikit' (/home/usuario/Projects/llm-zoomcamp-lessons/.venv/lib/python3.14/site-packages/toyaikit/__init__.py)